In [4]:
# Extended Gabidulin Code Decoder - Core Implementation
# Based on Song et al.'s original code

from sage.all import *
import math
import sys

def coefficient_vector(elt, m):
    """Return coefficient vector of an element in GF(q^m) over GF(q)."""
    coeffs = elt.polynomial().list()
    coeffs += [0] * (m - len(coeffs))
    return coeffs

def vector_rank(v, q, m):
    """Compute the rank of a vector in GF(q^m)^n over GF(q)."""
    rows = [coefficient_vector(elt, m) for elt in v]
    return matrix(GF(q), rows).rank()

def random_small_vector_generation(Extension, Length, Weight, Fqm):
    """Generate a vector of exact rank Weight in F_{q^m}^{Length}."""
    q = Fqm.base_ring().order()
    Fq = GF(q)
    
    # Create Weight × Extension matrix of full rank
    B = random_matrix(Fq, Weight, Extension)
    while B.rank() != Weight:
        B = random_matrix(Fq, Weight, Extension)
    
    # Create Length × Weight matrix
    C = random_matrix(Fq, Length, Weight) * B
    
    # Convert rows to Fqm elements
    return vector(Fqm, [Fqm(list(C.row(i))) for i in range(Length)])

def monte_carlo_test(q, m, n, t, k, r, w, trials=1000):
    """Run Monte Carlo simulation for decoding failure rate."""
    print(f"\nRunning Monte Carlo DFR Test")
    print(f"Parameters: q={q}, m={m}, n={n}, t={t}, k={k}, r={r}, w={w}")
    print(f"Trials: {trials}")
    print("="*60)
    
    # Initialize finite field
    Fqm = GF(q**m, 'a')
    Frob = Fqm.frobenius_endomorphism()
    S = OrePolynomialRing(Fqm, Frob, 'x')
    
    # Generate fixed generator vector g
    g = random_small_vector_generation(m, n, min(t, m, n), Fqm)
    g_rank = vector_rank(g, q, m)
    print(f"Generator vector rank: {g_rank} (expected: {min(t, m, n)})")
    
    # Precompute A2 matrix
    g_monomials = [g[i]**(q**j) for i in range(n) for j in range(k+r)]
    A2 = matrix(Fqm, n, k+r, g_monomials)
    
    failures = 0
    
    for trial in range(1, trials + 1):
        # Generate random message polynomial
        f = S.random_element(degree=k-1)
        
        # Generate error vector
        e = random_small_vector_generation(m, n, w, Fqm)
        
        # Create received vector
        y = vector([f(g[i]) + e[i] for i in range(n)])
        
        # Construct matrices
        y_monomials = [y[i]**(q**j) for i in range(n) for j in range(r+1)]
        A1 = matrix(Fqm, n, r+1, y_monomials)
        
        A = block_matrix(Fqm, 1, 2, [A1, A2])
        
        # Check kernel dimension
        kernel_dim = A.right_kernel().dimension()
        expected_dim = r - w + 1
        
        if kernel_dim != expected_dim:
            failures += 1
        else:
            # Get solution from kernel
            kernel_basis = A.right_kernel().basis()
            Solution = kernel_basis[0]
            
            V = S(list(Solution)[0:r+1])
            N = S(list(Solution)[r+1:k+2*r+1])
            
            # Left division
            try:
                ff, remainder = N.left_quo_rem(-V)
                if remainder != 0:
                    failures += 1
                else:
                    # Verify recovery
                    c_recovered = vector([ff(g[i]) for i in range(n)])
                    e_recovered = y - c_recovered
                    if not (ff == f and e_recovered == e):
                        failures += 1
            except:
                failures += 1
        
        # Progress every 100 trials
        if trial % 100 == 0:
            current_dfr = float(failures) / trial  # Convert to float
            print(f"  Progress: {trial}/{trials}, failures: {failures}, DFR: {current_dfr:.6f}")
    
    failure_rate = float(failures) / trials  # Convert to float
    print(f"\nFinal Results:")
    print(f"  Successes: {trials - failures}/{trials}")
    print(f"  Failures:  {failures}/{trials}")
    print(f"  DFR: {failure_rate:.6f}")
    
    if failure_rate > 0:
        print(f"  Approx. 2^{math.log2(failure_rate):.2f}")
    
    return failure_rate

# ========== TEST CASES ==========

print("="*60)
print("Extended Gabidulin Code Decoding Failure Rate Test")
print("="*60)

# Test 1: Easy case (w=12) 
print("\nTest 1: Easy case (w=13)")
params1 = {
    'q': 2, 'm': 27, 'n': 41, 't': 27,
    'k': 9, 'r': 16, 'w': 13, 'trials': 10000
}
dfr1 = monte_carlo_test(**params1)


# Test 2: Medium case (w=5) 
print("\n" + "="*60)
print("\nTest 2: Medium case (w=14)")
params2 = {
    'q': 2, 'm': 27, 'n': 41, 't': 27,
    'k': 9, 'r': 16, 'w': 14, 'trials': 10000
}
dfr2 = monte_carlo_test(**params2)


# Test 3: Hard case (w=15) 
print("\n" + "="*60)
print("\nTest 3: Hard case (w=15)")
params3 = {
    'q': 2, 'm': 27, 'n': 41, 't': 27,
    'k': 9, 'r': 16, 'w': 15, 'trials': 10000
}
dfr3 = monte_carlo_test(**params3)


# Test 4: Extreme case (w=16) 
print("\n" + "="*60)
print("\nTest 4: Extreme case (w=16)")
params4 = {
    'q': 2, 'm': 27, 'n': 41, 't': 27,
    'k': 9, 'r': 16, 'w': 16, 'trials': 10000
}
dfr4 = monte_carlo_test(**params4)


Extended Gabidulin Code Decoding Failure Rate Test

Test 1: Easy case (w=13)

Running Monte Carlo DFR Test
Parameters: q=2, m=27, n=41, t=27, k=9, r=16, w=13
Trials: 10000
Generator vector rank: 27 (expected: 27)
  Progress: 100/10000, failures: 1, DFR: 0.010000
  Progress: 200/10000, failures: 1, DFR: 0.005000
  Progress: 300/10000, failures: 1, DFR: 0.003333
  Progress: 400/10000, failures: 1, DFR: 0.002500
  Progress: 500/10000, failures: 1, DFR: 0.002000
  Progress: 600/10000, failures: 1, DFR: 0.001667
  Progress: 700/10000, failures: 1, DFR: 0.001429
  Progress: 800/10000, failures: 1, DFR: 0.001250
  Progress: 900/10000, failures: 1, DFR: 0.001111
  Progress: 1000/10000, failures: 1, DFR: 0.001000
  Progress: 1100/10000, failures: 1, DFR: 0.000909
  Progress: 1200/10000, failures: 2, DFR: 0.001667
  Progress: 1300/10000, failures: 2, DFR: 0.001538
  Progress: 1400/10000, failures: 2, DFR: 0.001429
  Progress: 1500/10000, failures: 2, DFR: 0.001333
  Progress: 1600/10000, failure